# Stage 2.5 — old PPO model × heuristic upkeep

**Run these cells in the existing live Kaggle session containing your checkpoints.**
A new notebook session does not inherit another session's `/kaggle/working` files.
Do not restart the session just to use this notebook; copy its cells into the live editor if necessary.
Enable Internet and allow this notebook access to the Kaggle Secret **GITHUB_TOKEN**.

Default: **P final**, fresh stochastic JAX sampling, original BC-E opponent,
`E_LEGACY`, standard opening, official 1.32.7 engine, **16 games** (2 seeds × 2 seats × 4 arms).
Only the candidate's executor changes. The original repo, runs, datasets and
submission archives are preserved. No training or Kaggle submission occurs.
This does not reproduce the O-u10 fixed-personality archive's precomputed noise.

A fresh checkout is pinned to the published implementation commit. For this small
sequential screen JAX defaults to CPU in a child process; notebook accelerator
settings and installed JAX/JAXLIB are left untouched.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, subprocess, tempfile, time, signal, hashlib

ROOT = Path('/kaggle/working/interactive_curriculum_0acfe858')
CHECKPOINTS = {
    'P_final': ROOT / 'runs/P_bce_fullspace_lr3e5_from_Ou10_s43049/final.npz',
    'P_u5_rollout_source': ROOT / 'runs/P_bce_fullspace_lr3e5_from_Ou10_s43049/ppo_update_000003.npz',
    'O_u10_source': ROOT / 'runs/O_bce_fullspace_lr1e5_from_Nu10_s43048/ppo_update_000008.npz',
}
MODEL = 'P_final'
PPO_CHECKPOINT = CHECKPOINTS[MODEL]
BC_E_CHECKPOINT = Path('/kaggle/input/datasets/billll/v0-bc-e/best.pt')
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
BRANCH = 'codex/stage25-upkeep-ablation'
CODE_SHA = 'ea71b946f9eb190e17a2e953b34d32e36224d21c'
SEEDS = [144368101, 2112243121]  # exploratory replay seeds, not a held-out panel
MASTER_SEED = 25
VARIANTS = ['baseline', 'care', 'fertilizer', 'combined']
JAX_PLATFORM = 'cpu'  # set to None to let the child process use the runtime default

for p in (ROOT, ROOT / 'repo', PPO_CHECKPOINT, BC_E_CHECKPOINT):
    if not p.exists():
        raise FileNotFoundError(f'Missing {p}. Use the original live session or restore the checkpoint files first.')
RUN_ROOT = ROOT / ('stage25_upkeep_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
RUN_ROOT.mkdir(exist_ok=False)
REPO = RUN_ROOT / 'repo'
OUTPUT = RUN_ROOT / 'results'
print('Model:', MODEL)
print('Checkpoint:', PPO_CHECKPOINT)
print('Run folder:', RUN_ROOT)
print('Planned games:', len(SEEDS) * 2 * len(VARIANTS))


In [ ]:
# Authenticate without putting the token into a URL, notebook output or git config.
from kaggle_secrets import UserSecretsClient
_token = UserSecretsClient().get_secret(GITHUB_SECRET_NAME)
if not _token:
    raise RuntimeError('The GitHub secret is empty.')
try:
    with tempfile.TemporaryDirectory(prefix='stage25_git_') as tmp:
        askpass = Path(tmp) / 'askpass.py'
        askpass.write_text('#!' + sys.executable + '\n' +
                          'import os,sys\n' +
                          'print("x-access-token" if "username" in sys.argv[1].lower() else os.environ["STAGE25_GIT_TOKEN"])\n')
        askpass.chmod(0o700)
        git_env = {**os.environ, 'GIT_ASKPASS': str(askpass),
                   'GIT_TERMINAL_PROMPT': '0', 'STAGE25_GIT_TOKEN': _token}
        subprocess.run(['git', 'clone', '--depth', '10', '--single-branch', '--branch', BRANCH,
                        'https://github.com/BillXu21/Kaggriculture.git', str(REPO)],
                       env=git_env, check=True, timeout=180)
        subprocess.run(['git', 'checkout', '--detach', CODE_SHA], cwd=REPO,
                       env=git_env, check=True, timeout=30)
finally:
    _token = None
    if 'git_env' in globals():
        git_env.pop('STAGE25_GIT_TOKEN', None)
actual_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert actual_sha == CODE_SHA, (actual_sha, CODE_SHA)
print('Pinned source:', actual_sha)


In [ ]:
# Child-process environment: keep the existing notebook/runtime packages intact.
EVAL_ENV = os.environ.copy()
EVAL_ENV['PYTHONUNBUFFERED'] = '1'
EVAL_ENV['OMP_NUM_THREADS'] = '1'
EVAL_ENV['MKL_NUM_THREADS'] = '1'
EVAL_ENV['OPENBLAS_NUM_THREADS'] = '1'
EVAL_ENV['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
if JAX_PLATFORM:
    EVAL_ENV['JAX_PLATFORMS'] = JAX_PLATFORM
EVAL_ENV['PYTHONPATH'] = str(REPO)

guard = 'from oracle.provenance import require_official_modules; require_official_modules(); print("Official engine provenance passed")'
probe = subprocess.run([sys.executable, '-c', guard], cwd=REPO, env=EVAL_ENV,
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if probe.returncode:
    print('Existing official engine failed the pinned check; installing an isolated 1.32.7 copy.')
    ENGINE_PACKAGES = RUN_ROOT / 'official_packages'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--target',
                    str(ENGINE_PACKAGES), 'kaggle-environments==1.32.7'], check=True, timeout=180)
    EVAL_ENV['PYTHONPATH'] = os.pathsep.join([str(REPO), str(ENGINE_PACKAGES)])
subprocess.run([sys.executable, '-c', guard], cwd=REPO, env=EVAL_ENV, check=True)
# Do not pip-install requirements-jax.txt or upgrade JAX/JAXLIB on the TPU image.


In [ ]:
# Validate real checkpoint reconstruction and runtime BEFORE starting the panel.
preflight = r"""
import sys, json
import torch, jax, optax, pyarrow
from bc_manager_jax.checkpoint import load_torch_checkpoint
from bc_manager_jax.model import ManagerConfig
from rl_manager.ppo_checkpoint import load_ppo_checkpoint
from rl_manager.ppo_adapter import ppo_batched_policy_from_state
from rl_manager.ppo_policy import CurriculumMaskConfig
from tools.evaluate_stage25_upkeep import UpkeepFactory
params, metadata = load_torch_checkpoint(sys.argv[2], expected_e_history_version='E_LEGACY')
config = ManagerConfig(**metadata['model_config'])
state, meta = load_ppo_checkpoint(sys.argv[1], config=config, expected_e_history_version='E_LEGACY')
policy = ppo_batched_policy_from_state(state, config, deterministic=False,
    e_history_version='E_LEGACY', curriculum=CurriculumMaskConfig.from_json_dict(meta['curriculum']))
print(json.dumps({'jax':jax.__version__, 'devices':[str(d) for d in jax.devices()],
    'history':meta['e_history_version'], 'curriculum':meta['curriculum'],
    'model_config':metadata['model_config'], 'policy':policy.identity.to_json_dict()}, indent=2))
print('Checkpoint preflight passed; no games run yet.')
"""
subprocess.run([sys.executable, '-c', preflight, str(PPO_CHECKPOINT), str(BC_E_CHECKPOINT)],
               cwd=REPO, env=EVAL_ENV, check=True)


## Run the paired screen

This cell stays in the foreground and prints each completed game. First inference
may compile without printing a game result. Each game is saved immediately.
Interrupting the cell stops its child process; partial results remain available.
To rerun, choose a new output directory rather than overwriting partial results.
The two source replay seeds make this a diagnostic screen, not promotion evidence.


In [ ]:
command = [sys.executable, '-m', 'tools.evaluate_stage25_upkeep',
           '--checkpoint', str(PPO_CHECKPOINT), '--e-checkpoint', str(BC_E_CHECKPOINT),
           '--output-dir', str(OUTPUT), '--backend', 'official',
           '--e-history-version', 'E_LEGACY', '--master-seed', str(MASTER_SEED),
           '--seeds', *map(str, SEEDS), '--variants', *VARIANTS]
(RUN_ROOT / 'launch.json').write_text(json.dumps({'command':command, 'cwd':str(REPO),
    'source_commit':CODE_SHA, 'jax_platform':JAX_PLATFORM}, indent=2))
print('Starting', len(SEEDS) * 2 * len(VARIANTS), 'games; logs:', RUN_ROOT / 'run.log', flush=True)
started = time.monotonic()
with (RUN_ROOT / 'run.log').open('w') as log:
    process = subprocess.Popen(command, cwd=REPO, env=EVAL_ENV, text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    try:
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end='', flush=True)
        returncode = process.wait()
    except BaseException:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
if returncode:
    raise RuntimeError(f'Evaluation exited {returncode}; inspect {RUN_ROOT / "run.log"}. Partial outputs preserved.')
print(f'Completed in {(time.monotonic()-started)/60:.1f} minutes.')


In [ ]:
import pandas as pd
from IPython.display import display, FileLink
rows = [json.loads(line) for line in (OUTPUT / 'games.jsonl').read_text().splitlines()]
games = pd.DataFrame(rows)
if len(games) != len(SEEDS) * 2 * len(VARIANTS):
    raise RuntimeError('Incomplete panel; do not interpret as a finished comparison.')
if not all(status == ['DONE', 'DONE'] for status in games['statuses']):
    raise RuntimeError('Invalid game status in panel.')
base = games[games.variant == 'baseline'][['seed','seat','bank','opponent_bank']]
paired = games.merge(base, on=['seed','seat'], suffixes=('', '_baseline'), validate='many_to_one')
paired['bank_delta'] = paired.bank - paired.bank_baseline
paired['margin_delta'] = (paired.bank-paired.opponent_bank) - (paired.bank_baseline-paired.opponent_bank_baseline)
paired['win'] = (paired.bank > paired.opponent_bank).astype(int)
summary = paired.groupby('variant', sort=False).agg(
    games=('bank','size'), mean_bank=('bank','mean'), median_bank=('bank','median'),
    mean_bank_delta=('bank_delta','mean'), min_bank_delta=('bank_delta','min'),
    mean_margin_delta=('margin_delta','mean'), wins=('win','sum'))
display(summary.round(1))
display(paired[['variant','seed','seat','bank','opponent_bank','bank_delta','margin_delta']])
summary.to_csv(OUTPUT / 'summary.csv')
paired.to_csv(OUTPUT / 'paired_games.csv', index=False)
# Persist diagnostics as a downloadable bundle; do not bundle checkpoints or secrets.
import shutil
bundle = shutil.make_archive(str(RUN_ROOT / 'stage25_upkeep_results'), 'zip', root_dir=OUTPUT)
display(FileLink(bundle))
print('Send back summary.csv, paired_games.csv, and the per-arm JSONs (or the ZIP).')
print('Four games per arm is a screen: confirm promising changes on more fresh seeds before adoption.')
